In [1]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecMonitor

### Add mbt-gym to path

In [2]:
import sys
sys.path.append("../")

In [3]:
from mbt_gym.agents.BaselineAgents import CarteaJaimungalMmAgent
from mbt_gym.gym.helpers.generate_trajectory import generate_trajectory
from mbt_gym.gym.StableBaselinesTradingEnvironment import StableBaselinesTradingEnvironment
from mbt_gym.gym.TradingEnvironment import TradingEnvironment
from mbt_gym.gym.wrappers import *
from mbt_gym.rewards.RewardFunctions import PnL, CjMmCriterion
from mbt_gym.stochastic_processes.midprice_models import BrownianMotionMidpriceModel
from mbt_gym.stochastic_processes.arrival_models import PoissonArrivalModel
from mbt_gym.stochastic_processes.fill_probability_models import ExponentialFillFunction
from mbt_gym.gym.ModelDynamics import LimitOrderModelDynamics

### Create market making environment

In [4]:
import sys
sys.path.append("../") # This version of the notebook is in the subfolder "notebooks" of the repo

import gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.integrate import quad

from copy import deepcopy


from mbt_gym.agents.BaselineAgents import *
from mbt_gym.gym.TradingEnvironment import TradingEnvironment
from mbt_gym.gym.helpers.generate_trajectory import generate_trajectory
from mbt_gym.gym.helpers.plotting import *
from mbt_gym.stochastic_processes.midprice_models import *
from mbt_gym.stochastic_processes.arrival_models import *
from mbt_gym.stochastic_processes.fill_probability_models import *
import torch
#print(torch.cuda.is_available())
#print(torch.cuda.get_device_name())
from mbt_gym.gym.ModelDynamics import LimitOrderModelDynamics
seed = 1


## Varying fad proportion (paramter q)

### Parameters

In [5]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 100
initial_inventory = 0
fads_proportion_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1]
# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
phi = 15
k = 1
gamma = 1
alpha=0.001
mu=0
big_phi=0.1

In [6]:
# OU parameters
u0 = 0.0
xi = 1.0

def m(t):
    return u0 * np.exp(-eta * t)

def v(t):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(q):
    c = gamma * sigma * q
    def integrand(t):
        return np.exp(c * m(t) + 0.5 * (c**2) * v(t))
    denom, _ = quad(integrand, 0.0, terminal_time)
    return (30.0 - phi * terminal_time) / denom

# compute psi for each fad proportion
psi_values = {q: compute_psi(q) for q in fads_proportion_values}

print("psi values:", psi_values)

psi values: {0.0: 15.0, 0.2: 14.985756598048642, 0.4: 14.943105475282236, 0.6: 14.872283180889763, 0.8: 14.773681637922692, 1: 14.647844682743013}


In [7]:
def get_as_env(num_trajectories:int = 1, fads_proportion:float = 0.6, psi:float = 15.0):
    midprice_model = BrownianMotionMidpriceModel(drift=mu, initial_price=initial_price,
                                                 volatility=sigma, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    intensity =  np.array([psi + phi, psi + phi])  # intensity for PoissonArrivalModel
    arrival_model = PoissonArrivalModel(intensity=intensity,
                                       step_size=terminal_time/n_steps,
                                       num_trajectories=num_trajectories)
    fill_probability_model = ExponentialFillFunction(fill_exponent=k, 
                                                     step_size=1/n_steps,
                                                     num_trajectories=num_trajectories)
    reward = CjMmCriterion(per_step_inventory_aversion=big_phi,
                           terminal_inventory_aversion=alpha,
                           terminal_time=terminal_time)
    LOtrader = LimitOrderModelDynamics(midprice_model=midprice_model, arrival_model=arrival_model,
                                        fill_probability_model=fill_probability_model,
                                        num_trajectories=num_trajectories)
    env_params = dict(terminal_time=terminal_time,
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [8]:
results_dict = {}
for q, psi_val in zip(fads_proportion_values, psi_values.values()):
    vec_env = get_as_env(
        num_trajectories=1000,
        fads_proportion=q,
        psi=psi_val
    )

    vec_as = CarteaJaimungalMmAgent(env=vec_env)

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(
        vec_env=vec_env, agent=vec_as
    )

    results_dict[(q, psi_val)] = dict(
        results=results,
        rewards=total_rewards,
        obs=observations
    )

In [9]:
header = f"{'Fads Prop':>10} | {'Psi':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))

for (fads_prop, psi_val), result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{fads_prop:10.2f} | {psi_val:10.4f} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15.2f} | {std_inv:13.2f}")


 Fads Prop |        Psi |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
-----------------------------------------------------------------------------------
      0.00 |    15.0000 |      21.33 |       4.99 |            0.07 |          3.00
      0.20 |    14.9858 |      21.32 |       5.01 |            0.07 |          3.00
      0.40 |    14.9431 |      21.27 |       5.00 |            0.07 |          2.99
      0.60 |    14.8723 |      21.23 |       4.99 |            0.07 |          2.99
      0.80 |    14.7737 |      21.17 |       4.99 |            0.06 |          2.99
      1.00 |    14.6478 |      21.07 |       4.97 |            0.05 |          2.97


## Varying eta

### Parameters

In [10]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 100
initial_inventory = 0

fads_proportion = 0.6 # p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta_values = [2.5, 5, 7.5, 10.0, 12.5]
phi = 15
k = 1
gamma = 1
alpha=0.001
mu=0
big_phi=0.1

In [11]:

# OU parameters
u0 = 0.0
xi = 1.0

def m(t, eta):
    return u0 * np.exp(-eta * t)

def v(t, eta):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(q, eta):
    c = gamma * sigma * q
    def integrand(t):
        return np.exp(c * m(t, eta) + 0.5 * (c**2) * v(t, eta))
    denom, _ = quad(integrand, 0.0, terminal_time)
    return (30.0 - phi * terminal_time) / denom

# compute psi for each eta
psi_values = {eta: compute_psi(fads_proportion, eta) for eta in eta_values}

# pretty print
for eta, val in psi_values.items():
    print(f"eta={eta:.1f} -> psi={val:.6f}")

eta=2.5 -> psi=14.572885
eta=5.0 -> psi=14.758861
eta=7.5 -> psi=14.832907
eta=10.0 -> psi=14.872283
eta=12.5 -> psi=14.896670


In [12]:
def get_as_env(num_trajectories:int = 1,eta:float=10, psi:float=15.0):
    midprice_model = BrownianMotionMidpriceModel(drift=mu, initial_price=initial_price,
                                                 volatility=sigma, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)

    intensity =  np.array([psi + phi, psi + phi])  # intensity for PoissonArrivalModel
    arrival_model = PoissonArrivalModel(intensity=intensity,
                                       step_size=terminal_time/n_steps,
                                       num_trajectories=num_trajectories)
    fill_probability_model = ExponentialFillFunction(fill_exponent=k, 
                                                     step_size=1/n_steps,
                                                     num_trajectories=num_trajectories)
    reward = CjMmCriterion(per_step_inventory_aversion=big_phi,
                           terminal_inventory_aversion=alpha,
                           terminal_time=terminal_time)
    LOtrader = LimitOrderModelDynamics(midprice_model=midprice_model, arrival_model=arrival_model,
                                        fill_probability_model=fill_probability_model,
                                        num_trajectories=num_trajectories)
    env_params = dict(terminal_time=terminal_time,
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [13]:
results_dict = {}
for eta in eta_values:
    vec_env = get_as_env(num_trajectories=1000, eta=eta, psi=psi_values[eta])

    vec_as = CarteaJaimungalMmAgent(env=vec_env)

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[eta] = dict(results=results, rewards=total_rewards, obs=observations)

In [14]:
header = f"{'Eta':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for etas, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{etas:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

       Eta |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
       2.5 |      21.02 |       4.97 |           0.045 | 2.973041372063295
         5 |      21.15 |       4.99 |           0.053 | 2.985999162759427
       7.5 |      21.20 |       4.99 |           0.066 | 2.987246893043827
      10.0 |      21.23 |       4.99 |           0.066 | 2.9869121178903137
      12.5 |      21.24 |       4.99 |           0.066 | 2.9852376789796824


## Varying gamma parameter

### Parameters

In [15]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 100
initial_inventory = 0
fads_proportion_values = 0.6# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
phi = 15
k = 1
gamma_values = [0, 1, 2, 3]
alpha=0.001
mu=0
big_phi=0.1

In [16]:
# OU parameters
u0 = 0.0
xi = 1.0

def m(t, eta):
    return u0 * np.exp(-eta * t)

def v(t, eta):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(q, eta, gamma):
    c = gamma * sigma * q
    def integrand(t):
        return np.exp(c * m(t, eta) + 0.5 * (c**2) * v(t, eta))
    denom, _ = quad(integrand, 0.0, terminal_time)
    return (30.0 - phi * terminal_time) / denom

# compute psi for each gamma
psi_values = {gamma: compute_psi(fads_proportion, eta, gamma) for gamma in gamma_values}

# pretty print
for gamma, val in psi_values.items():
    print(f"gamma={gamma} -> psi={val:.6f}")

gamma=0 -> psi=15.000000
gamma=1 -> psi=14.872283
gamma=2 -> psi=14.495463
gamma=3 -> psi=13.888033


In [17]:
def get_as_env(num_trajectories:int = 1,gamma:float=1, psi:float=15.0):
    midprice_model = BrownianMotionMidpriceModel(drift=mu, initial_price=initial_price,
                                                 volatility=sigma, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    intensity =  np.array([psi + phi, psi + phi])  # intensity for PoissonArrivalModel
    arrival_model = PoissonArrivalModel(intensity=intensity,
                                       step_size=terminal_time/n_steps,
                                       num_trajectories=num_trajectories)
    fill_probability_model = ExponentialFillFunction(fill_exponent=k, 
                                                     step_size=1/n_steps,
                                                     num_trajectories=num_trajectories)
    reward = CjMmCriterion(per_step_inventory_aversion=big_phi,
                           terminal_inventory_aversion=alpha,
                           terminal_time=terminal_time)
    LOtrader = LimitOrderModelDynamics(midprice_model=midprice_model, arrival_model=arrival_model,
                                        fill_probability_model=fill_probability_model,
                                        num_trajectories=num_trajectories)
    env_params = dict(terminal_time=terminal_time,
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [18]:
results_dict = {}
for gamma in gamma_values:
    vec_env = get_as_env(num_trajectories=1000, gamma=gamma, psi=psi_values[gamma])

    vec_as = CarteaJaimungalMmAgent(env=vec_env)

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[gamma] = dict(results=results, rewards=total_rewards, obs=observations)

In [19]:
header = f"{'Gamma':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for gamma, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{gamma:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

     Gamma |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
         0 |      21.33 |       4.99 |           0.065 | 2.9977950230127477
         1 |      21.23 |       4.99 |           0.066 | 2.9869121178903137
         2 |      20.95 |       4.97 |            0.06 | 2.9800000000000004
         3 |      20.54 |       4.95 |            0.03 | 2.974743686437539


## Varying Informed trader proportion (psi and phi)

### Parameters

In [20]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 100
initial_inventory = 0
fads_proportion = 0.6
# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
k = 1
gamma = k
alpha=0.001
mu=0
big_phi=0.1

In [21]:
# OU parameters
u0 = 0.0
xi = 1.0

def m(t):
    return u0 * np.exp(-eta * t)

def v(t):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(phi):
    c = gamma * sigma * q
    def integrand(t):
        return np.exp(c * m(t) + 0.5 * (c**2) * v(t))
    denom, _ = quad(integrand, 0.0, terminal_time)
    return (30.0 - phi * terminal_time) / denom

def phi_from_informed(perc_informed):
    """
    Given the percentage of informed traders (0-100),
    compute phi according to the paper.
    """
    return 30 * (1 - perc_informed / 100.0)
def psi_from_informed(perc_informed):
    """
    Given percentage of informed traders, compute phi and psi.
    """
    phi = phi_from_informed(perc_informed)
    psi = compute_psi(phi)  # your eq (61) implementation

    return phi, psi

In [22]:
# Build the dictionary
def build_phi_psi_dict(percentages):
    results = {}
    for perc in percentages:
        phi, psi = psi_from_informed(perc)
        results[perc] = {"phi": phi, "psi": psi}
    return results

# Example usage
percentages = [0, 25, 50, 75, 100]
phi_psi_dict = build_phi_psi_dict(percentages)
for perc, vals in phi_psi_dict.items():
    print(f"{perc}% informed → phi = {vals['phi']:.2f}, psi = {vals['psi']:.4f}")



0% informed → phi = 30.00, psi = 0.0000
25% informed → phi = 22.50, psi = 7.3239
50% informed → phi = 15.00, psi = 14.6478
75% informed → phi = 7.50, psi = 21.9718
100% informed → phi = 0.00, psi = 29.2957


In [23]:
def get_as_env(num_trajectories:int = 1, phi:float=15, psi:float=15):
    midprice_model = BrownianMotionMidpriceModel(drift=mu, initial_price=initial_price,
                                                 volatility=sigma, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    intensity =  np.array([phi+psi, phi+psi])
    arrival_model = PoissonArrivalModel(intensity=intensity,
                                       step_size=terminal_time/n_steps,
                                       num_trajectories=num_trajectories)
    fill_probability_model = ExponentialFillFunction(fill_exponent=k, 
                                                     step_size=1/n_steps,
                                                     num_trajectories=num_trajectories)
    reward = CjMmCriterion(per_step_inventory_aversion=big_phi,
                           terminal_inventory_aversion=alpha,
                           terminal_time=terminal_time)
    LOtrader = LimitOrderModelDynamics(midprice_model=midprice_model, arrival_model=arrival_model,
                                        fill_probability_model=fill_probability_model,
                                        num_trajectories=num_trajectories)
    env_params = dict(terminal_time=terminal_time,
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [24]:
results_dict = {}
for perc, vals in phi_psi_dict.items():
    phi, psi = vals['phi'], vals['psi']
    
    # Set up environment and agent
    vec_env = get_as_env(num_trajectories=1000, phi=phi, psi=psi)
    vec_as = CarteaJaimungalMmAgent(env=vec_env)
    
    # Generate trajectory and results
    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)
    
    # Store in results dictionary keyed by percentage
    results_dict[perc] = {
        "results": results,
        "rewards": total_rewards,
        "obs": observations
    }

In [25]:
header = f"{'Perc':>6} | {'Phi':>8} | {'Psi':>8} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))

for perc, result in results_dict.items():
    phi, psi = phi_psi_dict[perc]['phi'], phi_psi_dict[perc]['psi']
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    
    print(f"{perc:6}% | {phi:8.2f} | {psi:8.4f} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")


  Perc |      Phi |      Psi |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------------------------
     0% |    30.00 |   0.0000 |      21.33 |       4.99 |           0.065 | 2.9977950230127477
    25% |    22.50 |   7.3239 |      21.19 |       4.99 |           0.066 | 2.9875816306839216
    50% |    15.00 |  14.6478 |      21.07 |       4.97 |           0.049 | 2.9679284021013714
    75% |     7.50 |  21.9718 |      20.94 |       4.97 |           0.058 | 2.979368389440957
   100% |     0.00 |  29.2957 |      20.81 |       4.97 |           0.046 | 2.9913013890278592
